# 📘 Notebook: Reconciliation of LightGBM vs TFT Forecasts (Top Series)

## 🧾 Cell 1 — Notebook Title (Markdown)

# Reconciliation of LightGBM vs TFT Forecasts  
## Focus on Top Revenue and Volume Time Series

This notebook reconciles forecasts produced by:
- **LightGBM (baseline, validation/backtest)**
- **Temporal Fusion Transformer (TFT, SOTA, true future forecast)**

The comparison focuses on:
- Top-N revenue series
- Top-N volume series

### Outputs
- Per-series reconciliation summary
- Monthly TFT forecasts with trends
- LightGBM validation metrics (MAE, WAPE)
- Excel and CSV exports for planning & reporting


### ⚙️ Cell 2 — Imports & Configuration

In [15]:
from __future__ import annotations

import numpy as np
import pandas as pd
from pathlib import Path


### ⚙️ Cell 3 — User Configuration (replaces argparse)

In [16]:
# -----------------------------
# File paths
# -----------------------------
HISTORY_FILE = "data/training/training_data_final.csv"
TFT_FILE = "outputs/tft_forecast_12m.csv"
LGBM_FILE = "outputs/baseline_lightgbm_forecast_results.csv"

# -----------------------------
# Output
# -----------------------------
OUT_DIR = Path("reconcile_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Selection parameters
# -----------------------------
TOP_N_REVENUE = 200
TOP_N_QTY = 200

# -----------------------------
# Trend detection
# -----------------------------
TREND_PCT_THRESHOLD = 0.10
TREND_MIN_MEAN = 1e-6

# -----------------------------
# Column names
# -----------------------------
TFT_POINT_COL = "q50"
TFT_LOW_COL = "q10"
TFT_HIGH_COL = "q90"

LGBM_PRED_COL = "forecast_qty"
LGBM_ACTUAL_COL = "y_qty"

CLIP_NONNEGATIVE = True


### 📈 Cell 4 — Trend Detection Utility

In [17]:
def trend_from_series(values: pd.Series,
                      pct_threshold: float,
                      min_mean: float) -> str:
    """
    Determine trend: increasing / decreasing / stable
    """
    s = values.dropna().astype(float)
    if len(s) < 2:
        return "stable"

    mean = float(s.mean())
    if abs(mean) < min_mean:
        return "stable"

    first, last = float(s.iloc[0]), float(s.iloc[-1])
    pct = (last - first) / abs(mean)

    if pct > pct_threshold:
        return "increasing"
    if pct < -pct_threshold:
        return "decreasing"
    return "stable"


### 📥 Cell 5 — Load TFT Forecast (Future 12 Months)

In [18]:
tft = pd.read_csv(TFT_FILE, parse_dates=["Date"])

assert "series_id" in tft.columns, "Missing series_id in TFT file"
assert TFT_POINT_COL in tft.columns, "Missing point forecast column"

forecast_start = tft["Date"].min()
forecast_end = tft["Date"].max()

forecast_start, forecast_end


(Timestamp('2024-12-01 00:00:00'), Timestamp('2025-11-01 00:00:00'))

### 🕒 Cell 6 — Define Historical Window (Last 12 Months)

In [19]:
hist_window_start = forecast_start - pd.DateOffset(months=12)
hist_window_end = forecast_start - pd.DateOffset(months=1)

hist_window_start, hist_window_end


(Timestamp('2023-12-01 00:00:00'), Timestamp('2024-11-01 00:00:00'))

### 📦 Cell 7 — Load Historical Data

In [20]:
hist = pd.read_csv(
    HISTORY_FILE,
    parse_dates=["Date"],
    usecols=[
        "customer_id", "product_id",
        "customer_name", "product_group", "product_name",
        "Date", "ordered_qty", "order_amount"
    ]
)

hist["series_id"] = (
    hist["customer_id"].astype(str) + "_" +
    hist["product_id"].astype(str)
)


C:\Users\chonchol\AppData\Local\Temp\ipykernel_12260\2881276875.py:1: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  hist = pd.read_csv(


### 🥇 Cell 8 — Identify Top Revenue & Volume Series

In [21]:
hist12 = hist.loc[
    (hist["Date"] >= hist_window_start) &
    (hist["Date"] <= hist_window_end)
]

agg = hist12.groupby("series_id", as_index=False).agg(
    hist_revenue=("order_amount", "sum"),
    hist_qty=("ordered_qty", "sum"),
    customer_id=("customer_id", "first"),
    product_id=("product_id", "first"),
    customer_name=("customer_name", "first"),
    product_name=("product_name", "first"),
    product_group=("product_group", "first"),
)

agg["rev_rank"] = agg["hist_revenue"].rank(ascending=False)
agg["qty_rank"] = agg["hist_qty"].rank(ascending=False)

agg["is_top_revenue"] = agg["rev_rank"] <= TOP_N_REVENUE
agg["is_top_qty"] = agg["qty_rank"] <= TOP_N_QTY
agg["is_top_series"] = agg["is_top_revenue"] | agg["is_top_qty"]

top_series = agg[agg["is_top_series"]].copy()
len(top_series)


257

### 🔮 Cell 9 — Filter TFT to Top Series & Add Time Fields

In [22]:
tft_top = tft.merge(top_series[["series_id"]], on="series_id")

tft_top["Year"] = tft_top["Date"].dt.year
tft_top["Month"] = tft_top["Date"].dt.month


### 📉 Cell 10 — TFT Trend Detection

In [23]:
trend_tft = (
    tft_top.sort_values(["series_id", "Date"])
           .groupby("series_id")[TFT_POINT_COL]
           .apply(lambda s: trend_from_series(
               s, TREND_PCT_THRESHOLD, TREND_MIN_MEAN))
           .reset_index(name="trend_tft")
)


### 📥 Cell 11 — Load LightGBM Validation Data

In [24]:
lgbm = pd.read_csv(LGBM_FILE, parse_dates=["Date"])

lgbm["series_id"] = (
    lgbm["customer_id"].astype(str) + "_" +
    lgbm["product_id"].astype(str)
)

lgbm_top = lgbm.merge(top_series[["series_id"]], on="series_id")

if CLIP_NONNEGATIVE:
    lgbm_top[LGBM_PRED_COL] = lgbm_top[LGBM_PRED_COL].clip(lower=0)

lgbm_top["Year"] = lgbm_top["Date"].dt.year
lgbm_top["Month"] = lgbm_top["Date"].dt.month


### 📊 Cell 12 — LightGBM Trend & Performance Metrics

In [25]:
trend_lgbm = (
    lgbm_top.sort_values(["series_id", "Date"])
            .groupby("series_id")[LGBM_PRED_COL]
            .apply(lambda s: trend_from_series(
                s, TREND_PCT_THRESHOLD, TREND_MIN_MEAN))
            .reset_index(name="trend_lgbm")
)

perf = None
if LGBM_ACTUAL_COL in lgbm_top.columns:
    lgbm_top["abs_error"] = (
        lgbm_top[LGBM_PRED_COL] - lgbm_top[LGBM_ACTUAL_COL]
    ).abs()

    perf = lgbm_top.groupby("series_id", as_index=False).agg(
        lgbm_mae=("abs_error", "mean"),
        lgbm_wape_num=("abs_error", "sum"),
        lgbm_wape_den=(LGBM_ACTUAL_COL, "sum"),
    )

    perf["lgbm_wape"] = (
        perf["lgbm_wape_num"] /
        perf["lgbm_wape_den"].replace({0: np.nan})
    )


### 🧩 Cell 13 — Series-Level Summary Table

In [26]:
summary = (
    top_series
        .merge(trend_tft, on="series_id", how="left")
        .merge(trend_lgbm, on="series_id", how="left")
)

if perf is not None:
    summary = summary.merge(
        perf[["series_id", "lgbm_mae", "lgbm_wape"]],
        on="series_id",
        how="left"
    )

summary["hist_window_start"] = hist_window_start.date().isoformat()
summary["hist_window_end"] = hist_window_end.date().isoformat()
summary["forecast_start"] = forecast_start.date().isoformat()
summary["forecast_end"] = forecast_end.date().isoformat()

summary["trend_flag"] = summary["trend_tft"].fillna("stable")

summary.head()


,series_id,hist_revenue,hist_qty,customer_id,product_id,customer_name,product_name,product_group,rev_rank,qty_rank,...,is_top_series,trend_tft,trend_lgbm,lgbm_mae,lgbm_wape,hist_window_start,hist_window_end,forecast_start,forecast_end,trend_flag
0,2029_11500AO,2611.2,20.0,2029,11500AO,LLP FARM MACHINERY GROUP,0pp07306303504000600 63/35-400 a600 Oranssi,5002,306.0,158.0,...,True,stable,stable,3.340626,2.004376,2023-12-01,2024-11-01,2024-12-01,2025-11-01,stable
1,2029_11501DO,15589.0,100.0,2029,11501DO,LLP FARM MACHINERY GROUP,0pp07306303504000660 63/35-400-a660 ge35 Oranssi,5002,68.0,40.0,...,True,stable,stable,16.380001,1.965600,2023-12-01,2024-11-01,2024-12-01,2025-11-01,stable
2,2029_11508AM,3072.3,30.0,2029,11508AM,LLP FARM MACHINERY GROUP,0pp06005003001500350 50/30-150 musta,5002,269.0,108.0,...,True,stable,stable,5.101650,2.040660,2023-12-01,2024-11-01,2024-12-01,2025-11-01,stable
3,2029_11513M,11520.0,60.0,2029,11513M,LLP FARM MACHINERY GROUP,"Sylinteri 80/50-550 r3/8""ge35 Musta",5002,92.0,66.5,...,True,stable,stable,9.305002,1.861000,2023-12-01,2024-11-01,2024-12-01,2025-11-01,stable
4,2029_11514M,2366.8,20.0,2029,11514M,LLP FARM MACHINERY GROUP,Sylinteri 50/25-200 ge20 a398 RAL9005,5002,322.0,158.0,...,True,stable,stable,3.311084,1.986651,2023-12-01,2024-11-01,2024-12-01,2025-11-01,stable


### 🧩 Cell 14 — Series-Level Prediction

In [27]:

# --- Add TFT predicted quantities into Series_Summary (series-level aggregates) ---

# Make sure TFT top is sorted
tft_sorted = tft_top.sort_values(["series_id", "Date"]).copy()

# If you want non-negative predictions in summary:
if CLIP_NONNEGATIVE:
    for c in [TFT_POINT_COL, TFT_LOW_COL, TFT_HIGH_COL]:
        if c in tft_sorted.columns:
            tft_sorted[c] = tft_sorted[c].clip(lower=0)

# Series-level aggregates
tft_series_pred = (
    tft_sorted.groupby("series_id", as_index=False)
    .apply(lambda g: pd.Series({
        "pred_qty_m1": float(g[TFT_POINT_COL].iloc[0]),
        "pred_qty_total_12m": float(g[TFT_POINT_COL].sum()),
        "pred_qty_avg_12m": float(g[TFT_POINT_COL].mean()),
        "pred_qty_peak_12m": float(g[TFT_POINT_COL].max()),
        "pred_qty_peak_date": g.loc[g[TFT_POINT_COL].idxmax(), "Date"],
        "pred_pi_width_avg": float((g[TFT_HIGH_COL] - g[TFT_LOW_COL]).mean()) if (TFT_LOW_COL in g and TFT_HIGH_COL in g) else np.nan,
    }))
    .reset_index(drop=True)
)

# Merge into summary table
summary = summary.merge(tft_series_pred, on="series_id", how="left")

C:\Users\chonchol\AppData\Local\Temp\ipykernel_12260\574850371.py:15: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({


### 💾 Cell 14 — Save Outputs (Excel + CSV)

In [28]:
summary_csv = OUT_DIR / "reconcile_series_summary_v2.csv"
tft_csv = OUT_DIR / "reconcile_top_series_tft_forecasts_v2.csv"
lgbm_csv = OUT_DIR / "reconcile_top_series_lgbm_validation_v2.csv"
xlsx_path = OUT_DIR / "reconcile_top_series_v2.xlsx"

summary.to_csv(summary_csv, index=False)
tft_top.to_csv(tft_csv, index=False)
lgbm_top.to_csv(lgbm_csv, index=False)

with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
    summary.to_excel(writer, sheet_name="Series_Summary", index=False)
    tft_top.to_excel(writer, sheet_name="TFT_Forecast_12M", index=False)
    lgbm_top.to_excel(writer, sheet_name="LGBM_Validation", index=False)

xlsx_path


WindowsPath('reconcile_outputs/reconcile_top_series_v2.xlsx')